In [18]:
# %pip install soundfile
# !pip install -q kaggle
!pip install -q wandb

In [ ]:
# import soundfile as sf
# print(sf.__version__)

## Libraries Declaration

In [1]:
from __future__ import annotations
from typing import Any, Dict, List, Optional, Tuple, Iterator, Sequence
from pathlib import Path
import json
import re
import wave
import numpy as np
import soundfile as sf
import os

import torch
from torch import Tensor, nn
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import DataLoader, Dataset
from torch.utils.data import BatchSampler
import time

from collections import defaultdict


## Download dataset
- The dataset is stored in the temporary Colab filesystem
- The /content directory is from Colab. This code is run in Colab

In [ ]:
# %cd /content

# !git clone --filter=blob:none --no-checkout \
#     https://github.com/microsoft/AEC-Challenge.git

# %cd /content/AEC-Challenge

# !git sparse-checkout init --cone

# !git sparse-checkout set \
#     datasets/synthetic/nearend_mic_signal \
#     datasets/synthetic/nearend_speech

# !git checkout main


## Upload dataset to Kaggle

In [ ]:
# import json
# from pathlib import Path

# dataset_dir = Path("/content/erb_store")

# metadata = {
#     "title": "Speech Dataset",
#     "id": "quanninhhoang/erb-speech-dataset",
#     "licenses": [{"name": "CC0-1.0"}]
# }

# with open(dataset_dir / "dataset-metadata.json", "w") as f:
#     json.dump(metadata, f)

In [ ]:
# !kaggle datasets create -p /content/erb_store --dir-mode zip

## Verify dataset

In [ ]:
# input_dir = Path(
#     "/content/AEC-Challenge/datasets/synthetic/nearend_mic_signal"
# )

# target_dir = Path(
#     "/content/AEC-Challenge/datasets/synthetic/nearend_speech"
# )

# print(len(list(input_dir.glob("*.wav"))), "input files")
# print(len(list(target_dir.glob("*.wav"))), "target files")

## Mount google drive to server(e.g, GPU T4)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## ERB filter bank

In [2]:
def load_mono(path):
    wav, sr = torchaudio.load(path)    # sr: sample rate
    wav = wav.mean(dim=0)
    return wav, sr

def stft_magnitude(wav, device, n_fft=512, hop_length=128, win_length=512):
    window = torch.hann_window(win_length, device=device)
    wav = wav.to(device)

    spec = torch.stft(
        wav,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=window,
        center=True,
        return_complex=True,
    )

    # [freq, time] -> [time, freq]
    return spec.transpose(0, 1)

def hz_to_erb(freq):
    return 21.4 * torch.log10(1.0 + 0.00437 * freq)

def erb_to_hz(erb):
    return (10 ** (erb / 21.4) - 1.0) / 0.00437

def make_erb_filterbank(sample_rate, n_fft, erb_bins, low_freq=0.0, high_freq=None):
    if high_freq is None:
        high_freq = sample_rate / 2

    erb_edges = torch.linspace(
        hz_to_erb(torch.tensor(low_freq)),
        hz_to_erb(torch.tensor(high_freq)),
        erb_bins + 2,
    )

    hz_edges = erb_to_hz(erb_edges)
    fft_freqs = torch.linspace(0.0, sample_rate / 2, n_fft // 2 + 1)

    filterbank = torch.zeros(erb_bins, n_fft // 2 + 1)

    for i in range(erb_bins):
        left = hz_edges[i]
        center = hz_edges[i + 1]
        right = hz_edges[i + 2]

        rising = (fft_freqs - left) / (center - left)
        falling = (right - fft_freqs) / (right - center)

        filterbank[i] = torch.minimum(rising, falling).clamp_min(0.0)
    return filterbank

# def wav_to_erb(path, sample_rate, filterbank, n_fft=512, hop_length=128, win_length=512):
#     wav, sr = load_mono(path)

#     if sr != sample_rate:
#         wav = torchaudio.functional.resample(wav, orig_freq=sr, new_freq=sample_rate)

#     magnitude = stft_magnitude(wav, n_fft=n_fft, hop_length=hop_length, win_length=win_length)
#     erb = magnitude @ filterbank.T
#     erb = torch.log1p(erb)
#     return erb 

def wav_pair_to_features(input_path: str | Path, target_path: str | Path, *, sample_rate: int, n_fft: int, hop_length: int, win_length: int, filterbank: Tensor, device) -> Dict[str, Tensor]:
    input_wav, input_sr = load_mono(input_path)
    target_wav, target_sr = load_mono(target_path)

    if input_sr != sample_rate:
        input_wav = torchaudio.functional.resample(input_wav, input_sr, sample_rate)
    if target_sr != sample_rate:
        target_wav = torchaudio.functional.resample(target_wav, target_sr, sample_rate)

    sample_count = min(input_wav.numel(), target_wav.numel())
    input_wav = input_wav[:sample_count]
    target_wav = target_wav[:sample_count]

    input_spec = stft_magnitude(input_wav, device, n_fft, hop_length, win_length)
    target_spec = stft_magnitude(target_wav, device, n_fft, hop_length, win_length)
    frame_count = min(input_spec.shape[0], target_spec.shape[0])
    input_spec = input_spec[:frame_count].contiguous()
    target_spec = target_spec[:frame_count].contiguous()

    input_power = input_spec.abs().square()
    target_power = target_spec.abs().square()
    input_erb = torch.log1p(input_power @ filterbank.to(device).T).float()
    target_erb = torch.log1p(target_power @ filterbank.to(device).T).float()

    record = {
        "input_erb": input_erb.cpu().half(),
        "target_erb": target_erb.cpu().half(),
        "input_spec": input_spec.to(torch.complex64).cpu(),
        "target_spec": target_spec.to(torch.complex64).cpu(),
    }

    del input_wav
    del target_wav
    del input_spec
    del target_spec
    del input_erb
    del target_erb

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return record

def build_erb_store(input_dir: str | Path, 
                    target_dir: str | Path, 
                    output_dir: str | Path, 
                    *, 
                    sample_rate: int = 16000, 
                    n_fft: int = 512, 
                    hop_length: int = 128, 
                    win_length: int = 512, 
                    erb_bins: int = 32,
                    device) -> None:
    input_dir = Path(input_dir)
    target_dir = Path(target_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    input_files = sorted(p for p in input_dir.rglob("*") if p.is_file() and p.suffix.lower() == ".wav")
    target_files = sorted(p for p in target_dir.rglob("*") if p.is_file() and p.suffix.lower() == ".wav")

    if not input_files:
        raise FileNotFoundError(f"No WAV files found under {input_dir}")
    if not target_files:
        raise FileNotFoundError(f"No WAV files found under {target_dir}")

    target_by_id: Dict[str, Path] = {}
    for target_path in target_files:
        match = re.search(r"(?:^|_)fileid_(.+)\.wav$", target_path.name)
        if match:
            target_by_id[match.group(1)] = target_path

    filterbank = make_erb_filterbank(sample_rate, n_fft, erb_bins)
    index_rows: List[Dict[str, Any]] = []
    shard_size = 200
    shard = []
    shard_id = 0

    for record_id, input_path in enumerate(input_files):
        if record_id % 10 == 0:
            print(
                f"Processing {record_id}/{len(input_files)}",
                flush=True,
            )
        target_path = target_dir / input_path.name
        if not target_path.exists():
            match = re.search(r"(?:^|_)fileid_(.+)\.wav$", input_path.name)
            target_path = target_by_id.get(match.group(1)) if match else None

        if target_path is None or not target_path.exists():
            raise FileNotFoundError(f"Missing target for {input_path.name} under {target_dir}")

        record = wav_pair_to_features(
            input_path,
            target_path,
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            win_length=win_length,
            filterbank=filterbank,
            device=device,
        )
        shard_index = len(shard)
        shard.append(record)

        index_rows.append(
            {
                "id": record_id,
                "input_file": str(input_path),
                "target_file": str(target_path),
                "shard_file": f"shard_{shard_id:04d}.pt",
                "shard_index": shard_index,
                "frames": int(record["input_erb"].shape[0]),
                "erb_bins": int(record["input_erb"].shape[1])
            }
        )

        if (len(shard) == shard_size):
            shard_path = (output_dir / f"shard_{shard_id:04d}.pt")
            torch.save(shard, shard_path)
            shard = []
            shard_id += 1
    if shard:
        shard_path = (output_dir / f"shard_{shard_id:04d}.pt")
        torch.save(shard, shard_path)
        
    index_path = output_dir / "index.jsonl"
    with index_path.open("w", encoding="utf-8") as handle:
        for row in index_rows:
            handle.write(json.dumps(row) + os.linesep)


## Loss Function

In [5]:
"""Return normalized inverse ERB weights with shape ``[F, E]``."""
def erb_synthesis_matrix(filterbank: Tensor, eps: float = 1e-8) -> Tensor:
    return filterbank.T / filterbank.sum(dim=0, keepdim=True).T.clamp_min(eps)

def apply_erb_gains(input_spec: Tensor, gains: Tensor, synthesis_matrix: Tensor) -> Tensor:
    if gains.ndim != 4 or gains.shape[1] != 1:
        raise ValueError("gains must have shape [B, 1, T, E]")
    """synthesis_matrix: [F, E], which converts values defined on 
    ERB bands back into values for individual FFT frequency bins.
       gains[:, 0]: makes [B, 1, T, E] -> [B, T, E]"""
    frequency_gain = gains[:, 0] @ synthesis_matrix.T    # -> [B, T, F]
    return input_spec * frequency_gain.to(input_spec.dtype)

def compressed_spectral_loss(predicted_spec: Tensor, target_spec: Tensor, power: float = 0.6) -> Tensor:
    pred_mag = predicted_spec.abs().clamp_min(1e-8).pow(power)
    target_mag = target_spec.abs().clamp_min(1e-8).pow(power)
    magnitude_loss = (pred_mag - target_mag).square()

    pred_phase = torch.angle(predicted_spec)
    target_phase = torch.angle(target_spec)

    pred_compressed_complex = (pred_mag * torch.exp(1j * pred_phase))
    target_compressed_complex = (target_mag * torch.exp(1j * target_phase))

    phase_aware_loss = (pred_compressed_complex - target_compressed_complex).abs().square()
    return (magnitude_loss + phase_aware_loss).mean()

## Set up layers

In [6]:
def channel_shuffle(x: Tensor, groups: int) -> Tensor:
    b, t, d = x.shape
    if d % groups != 0:
        raise ValueError("Feature dimension must be divisible by groups")

    features_per_group = d // groups
    x = x.view(b, t, groups, features_per_group)
    x = x.transpose(2, 3).contiguous()
    x = x.view(b, t, d)

    return x

class SeparableConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, *, kernel_size: Tuple[int, int] = (3, 2), stride: Tuple[int, int] = (1, 1), lookahead: int = 0) -> None:
        super().__init__()

        kt, kf = kernel_size

        if lookahead < 0 or lookahead > kt - 1:
            raise ValueError("lookahead must satisfy 0 <= lookahead <= kt - 1")

        time_left = kt - 1 - lookahead
        time_right = lookahead
        self.pad = (
            kf // 2, # left
            kf - 1 - kf // 2, # right
            time_left, # top
            time_right, # bottom
        )

        self.depthwise = nn.Conv2d(
            in_channels=in_channels,
            out_channels=in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=0,
            groups=in_channels,
            bias=False,
        )

        self.pointwise = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False,
        )

        self.norm = nn.BatchNorm2d(out_channels)
        self.act = nn.ReLU()
    def forward(self, x: Tensor) -> Tensor:
        x = F.pad(x, self.pad)
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.norm(x)
        x = self.act(x)
        return x
    
class SeparableTConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, *, scale_factor: int = 2, lookahead: int = 0) -> None:
        super().__init__()
        self.scale_factor = scale_factor

        self.conv = SeparableConv2d(in_channels, out_channels, kernel_size=(3, 2), lookahead=lookahead)

    def forward(self, x: Tensor) -> Tensor:
        x = F.interpolate(
            x, 
            scale_factor=(1, self.scale_factor), # [width, height] ~ [time, frequency]
            mode="nearest"
        )
        return self.conv(x)

class GroupedLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, groups: int = 8, shuffle: bool = True) -> None:
        super().__init__()

        if in_features % groups:
            raise ValueError("in_features must be divisible by groups")

        if out_features % groups:
            raise ValueError("out_features must be divisible by groups")

        self.groups = groups
        self.shuffle = shuffle

        self.in_per_group = in_features // groups
        self.out_per_group = out_features // groups

        self.layers = nn.ModuleList(
            [
                nn.Linear(
                    self.in_per_group,
                    self.out_per_group,
                )
                for _ in range(groups)
            ]
        )

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, T, D]

        chunks = x.split(self.in_per_group, dim=-1)

        output_chunks = []

        for layer, chunk in zip(self.layers, chunks):
            y = layer(chunk)
            output_chunks.append(y)

        x = torch.cat(output_chunks, dim=-1)

        if self.shuffle:
            x = channel_shuffle(x, self.groups)

        return x

class GroupedGRU(nn.Module):
    def __init__(self, input_size: int = 512, hidden_size: int = 512, groups: int = 8, shuffle: bool = True) -> None:
        super().__init__()
        if input_size % groups:
            raise ValueError("input_size must be divisible by groups")
        if hidden_size % groups:
            raise ValueError("hidden_size must be divisible by groups")

        self.groups = groups
        self.shuffle = shuffle

        self.input_per_group = input_size // groups
        self.hidden_per_groups = hidden_size // groups

        self.grus = nn.ModuleList(
            [
                nn.GRU(
                    input_size=self.input_per_group,
                    hidden_size=self.hidden_per_groups,
                    batch_first=True,
                )
                for _ in range(groups)
            ]
        )


    def forward(self, x: Tensor) -> Tensor:
        # [B, T, D]
        chunks = x.split(self.input_per_group, dim=-1)
        outputs = []

        for gru, chunk in zip(self.grus, chunks):
            # GRU returns [output, hidden]
            y, _ = gru(chunk)
            outputs.append(y)

        x = torch.cat(outputs, dim=-1)

        if self.shuffle:
            x = channel_shuffle(x, self.groups)

        return x

class GroupedGRUStack(nn.Module):
    def __init__(self, size: int = 512, groups: int = 8, num_layers: int = 3) -> None:
        super().__init__()

        self.layers = nn.ModuleList(
            [
                GroupedGRU(
                    input_size=size,
                    hidden_size=size,
                    groups=groups,
                    shuffle=True,
                )
                for _ in range(num_layers)
            ]
        )

    def forward(self, x: Tensor) -> Tensor:
        for layer in self.layers:
            x = layer(x)

        return x

class PConv(nn.Module):
    def __init__(self, channels: int = 64) -> None:
        super().__init__()

        self.conv = nn.Conv2d(channels, channels, kernel_size=1, bias=False)

    def forward(self, x: Tensor) -> Tensor:
        return self.conv(x)

## Implement ERB encoder, decoder and deep filter net

In [8]:
class ERBEncoder(nn.Module):
    def __init__(self, erb_bins: int = 32, channels: int = 64, hidden_size: int = 512, groups: int = 8, conv_lookahead: int = 2) -> None:
        super().__init__()

        if erb_bins % 8:
            raise ValueError("erb_bins must be divisible by 8")

        self.erb_bins = erb_bins
        self.channels = channels
        self.conv_lookahead = conv_lookahead
        lookahead0 = 1 if conv_lookahead > 0 else 0
        lookahead1 = 1 if conv_lookahead > 1 else 0
        lookahead2 = 1 if conv_lookahead > 2 else 0
        self.conv0 = SeparableConv2d(1, channels, lookahead=lookahead0)
        self.conv1 = SeparableConv2d(channels, channels, stride=(1, 2), lookahead=lookahead1) # B -> B / 2
        self.conv2 = SeparableConv2d(channels, channels, stride=(1, 2), lookahead=lookahead2)
        self.conv3 = SeparableConv2d(channels, channels, stride=(1, 2), lookahead=0)
        self.glinear = GroupedLinear(in_features=channels*erb_bins//8, out_features=hidden_size, groups=groups)
        self.gru = GroupedGRUStack(size=hidden_size, groups=groups, num_layers=3)

    def forward(self, x: Tensor) -> Tuple[Tensor, Tensor, Tensor, Tensor, Tensor]:
        x0 = self.conv0(x)
        x1 = self.conv1(x0)
        x2 = self.conv2(x1)
        x3 = self.conv3(x2)

        batch, channels, time, freq = x3.shape
        x = x3.permute(0, 2, 1, 3)
        x = x.reshape(batch, time, channels*freq)

        x = self.glinear(x)
        embedding = self.gru(x)

        return x0, x1, x2, x3, embedding

class ERBDecoder(nn.Module):
    def __init__(self, channels: int = 64, erb_bins: int = 32, hidden_size: int = 512) -> None:
        super().__init__()

        bottleneck_freq = erb_bins // 8
        bottleneck_size = channels * bottleneck_freq

        self.channels = channels
        self.bottelneck_freq = bottleneck_freq

        self.linear = GroupedLinear(hidden_size, bottleneck_size)

        self.p3 = PConv(channels)
        self.p2 = PConv(channels)
        self.p1 = PConv(channels)
        self.p0 = PConv(channels)

        self.up3 = SeparableTConv2d(channels, channels, lookahead=0)
        self.up2 = SeparableTConv2d(channels, channels, lookahead=0)
        self.up1 = SeparableTConv2d(channels, channels, lookahead=0)

        self.final_conv = SeparableConv2d(channels, 1, kernel_size=(3, 2), lookahead=0)
        self.output_activation = nn.Sigmoid()

    def forward(self, embedding: Tensor, x0: Tensor, x1: Tensor, x2: Tensor, x3: Tensor) -> Tensor:
        batch, time, _ = embedding.shape

        x = self.linear(embedding)

        x = x.reshape(batch, time, self.channels, self.bottelneck_freq)
        x = x.permute(0, 2, 1, 3)

        x = x + self.p3(x3)

        x = self.up3(x)
        x = x + self.p2(x2)

        x = self.up2(x)
        x = x + self.p1(x1)

        x = self.up1(x)
        x = x + self.p0(x0)

        gains = self.final_conv(x)
        gains = self.output_activation(gains)

        return gains

# class DFNet(nn.Module):
#     def __init__(self, df_bins: int, df_order: int, channels: int=64, hidden_size: int = 512, groups: int=8) -> None:
#         super().__init__()

#         self.df_bins = df_bins
#         self.df_order = df_order
#         self.channels = channels

#         self.conv0 = SeparableConv2d(1, channels, stride=(1, 1))
#         self.conv1 = SeparableConv2d(channels, channels, stride=(1, 2))
#         self.projection = GroupedLinear(channels*(df_bins // 2), hidden_size, groups=groups)
#         self.grus = GroupedGRUStack(size=hidden_size, groups=groups, num_layers=2)
#         self.pconv = PConv(channels)
#         self.output = nn.Linear(hidden_size, df_bins*df_order*2)
#         self.merge = nn.Linear(hidden_size*2, hidden_size)

#     def forward(self, complex_features: Tensor, encoder_embedding: Tensor) -> Tensor:
#         x = self.conv0(complex_features)
#         x = self.conv1(x)

#         batch, channels, time, freq = x.shape
#         x = x.permute(0, 2, 1, 3)
#         x = x.reshape(batch, time, channels * freq)

#         x = self.projection(x)
#         x = self.grus(x)

#         x = torch.cat([x, encoder_embedding], dim=-1)
#         x = self.merge(x)

#         coefficients = self.output(x)

#         coefficients = coefficients.reshape(batch, time, self.df_bins, self.df_order, 2)

#         return coefficients

        

## Deep Neural Network (DNN)

In [10]:
class DeepFilterNetDNN(nn.Module):
    """
    rb_features:
        [B, 1, T, B_erb]

    complex_features:
        [B, 1, T, F_df]

    erb_gains:
        [B, 1, T, B_erb]
    
    df_coefficients:
        [B, T, F_df, N, 2]
    """

    def __init__(self, erb_bins: int = 32, df_bins: int = 96, df_order: int = 5, channels: int = 64, hidden_size: int = 512, groups: int = 8, conv_lookahead: int = 2) -> None:
        super().__init__()
        self.conv_lookahead = conv_lookahead

        self.encoder = ERBEncoder(
            erb_bins=erb_bins,
            channels=channels,
            hidden_size=hidden_size,
            groups=groups,
            conv_lookahead=conv_lookahead,
        )

        self.decoder = ERBDecoder(
            channels=channels,
            erb_bins=erb_bins,
            hidden_size=hidden_size
        )

        # self.df_net = DFNet(
        #     df_bins=df_bins,
        #     df_order=df_order,
        #     channels=channels,
        #     hidden_size=hidden_size,
        #     groups=groups,
        # )

    # def forward(self, erb_features: Tensor, complex_features: Tensor) -> Tuple[Tensor, Tensor]:
    #     e0, e1, e2, e3, embedding = self.encoder(erb_features)
    #     gains = self.decoder(embedding, e0, e1, e2, e3)
    #     #df_coefficients = self.df_net(complex_features, embedding)

    #     return gains, df_coefficients

    def forward(self, erb_features: Tensor) -> Tensor:
            e0, e1, e2, e3, embedding = self.encoder(erb_features)
            gains = self.decoder(embedding, e0, e1, e2, e3)
            #df_coefficients = self.df_net(complex_features, embedding)
            return gains

## Build Dataset Object

In [3]:
class IndexedERBDataset(Dataset):
    def __init__(self, index_path: str | Path, *, segment_frames: Optional[int] = 256, random_crop: bool = True) -> None:
        index_path = Path(index_path)
        self.data_dir = index_path.parent
        self.rows = [
            json.loads(line)
            for line in Path(index_path).read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
        self.segment_frames = segment_frames
        self.random_crop = random_crop
        self.cached_shard_file = None
        self.cached_shard = None

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> Dict[str, Tensor]:
        row = self.rows[index]
        shard_file = row["shard_file"]

        if self.cached_shard_file != shard_file:
            shard_path = self.data_dir / shard_file
            self.cached_shard = torch.load(shard_path, map_location="cpu", weights_only=True)
            self.cached_shard_file = shard_file
        record = self.cached_shard[row["shard_index"]]
        total_frames = record["input_erb"].shape[0]
        frames = self.segment_frames or total_frames

        if total_frames >= frames:
            if self.random_crop:
                start = torch.randint(total_frames - frames + 1, ()).item()
            else:
                start = (total_frames - frames) // 2
            stop = start + frames
            result = {key: value[start:stop] for key, value in record.items()}
        else:
            result = {
                key: torch.nn.functional.pad(
                    value,
                    (0, 0, 0, frames - total_frames),
                )
                for key, value in record.items()
            }
        # Model input: [B, 1, T, E]; DataLoader adds B.
        result["input_erb"] = result["input_erb"].unsqueeze(0)
        result["target_erb"] = result["target_erb"].unsqueeze(0)
        return result

## Split Dataset

In [14]:
class DatasetSpliter(Dataset):
    def __init__(self, dataset: Dataset, indices: Sequence[int]):
        self.dataset = dataset
        self.indices = list(indices)
        
        # For Batch Sampler
        self.rows = [
            dataset.rows[index]
            for index in indices
        ]
        
    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        original_index = self.indices[index]
        return self.dataset[original_index]

In [15]:
def split_dataset(dataset, train_ratio=0.7, val_ratio=0.15, seed=1234):
    generator = torch.Generator()
    generator.manual_seed(seed)
    all_indices = torch.randperm(len(dataset), generator=generator).tolist()
    
    train_end = int(len(dataset) * train_ratio)
    val_end = train_end + int(len(dataset) * val_ratio)
    train_indices = all_indices[:train_end]
    val_indices = all_indices[:val_end]
    test_indices = all_indices[val_end:]

    train_dataset = DatasetSpliter(dataset, train_indices)
    val_dataset = DatasetSpliter(dataset, val_indices)
    test_dataset = DatasetSpliter(dataset, test_indices)
    return (train_dataset, val_dataset, test_dataset)
    

## BatchSampler

In [12]:
class ShardBatchSampler(BatchSampler):
    def __init__(self, dataset, batch_size: int, drop_last: bool = False, seed: int = 1234):
        self.dataset = dataset
        self.batch_size = batch_size
        self.drop_last = drop_last
        self.seed = seed
        self.epoch = 0
        self.indices_by_shard = defaultdict(list)
        for index, row in enumerate(dataset.rows):
            self.indices_by_shard[row["shard_file"]].append(index)
        self.shard_files = list(self.indices_by_shard.keys())

    def set_epoch(self, epoch: int):
        self.epoch = epoch

    def __iter__(self) -> Iterator[List[int]]:
        generator = torch.Generator()
        generator.manual_seed(self.seed + self.epoch)
        # Shuffle order of shard
        shard_order = torch.randperm(len(self.shard_files), generator=generator).tolist()
        for shard_position in shard_order:
            shard_file = self.shard_files[shard_position]
            shard_indices = self.indices_by_shard[shard_file]
            sample_order = torch.randperm(len(shard_indices), generator=generator).tolist()
            shuffled_indices = []
            for i in sample_order:
                shuffled_indices.append(shard_indices[i])
            # Create batches from a single shard
            for start in range(0, len(shuffled_indices), self.batch_size):
                batch = shuffled_indices[start:start + self.batch_size]
                if len(batch) < self.batch_size and self.drop_last:
                    continue
                yield batch

    def __len__(self) -> int:
        total_batches = 0

        for shard_indices in self.indices_by_shard.values():
            shard_length = len(shard_indices)
            if self.drop_last:
                total_batches += shard_length // self.batch_size
            else:
                total_batches += (shard_length + self.batch_size - 1) // self.batch_size
                
        return total_batches
            

## Define Parameters

In [11]:
sample_rate = 16000
n_fft = 512
hop_length = 128
win_length = 512
erb_bins = 32
batch_size = 8
epochs = 10
learning_rate = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
filterbank = make_erb_filterbank(sample_rate=sample_rate, n_fft=n_fft, erb_bins=erb_bins)


## Define Dataloader

In [17]:
indx_path = "/kaggle/input/datasets/quanninhhoang/erb-speech-dataset/index.jsonl"
dataset = IndexedERBDataset(index_path=indx_path)

train_dataset, val_dataset, test_dataset = split_dataset(
    dataset,
    train_ratio=0.70,
    val_ratio=0.15,
    seed=1234,
)

train_batch_sampler = ShardBatchSampler(
    train_dataset,
    batch_size=batch_size,
    drop_last=False,
    seed=1234,
)

val_batch_sampler = ShardBatchSampler(
    val_dataset,
    batch_size=batch_size,
    drop_last=False,
    seed=5678,
)

test_batch_sampler = ShardBatchSampler(
    test_dataset,
    batch_size=batch_size,
    drop_last=False,
    seed=9012,
)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_batch_sampler,
    num_workers=1,
    pin_memory=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_sampler=val_batch_sampler,
    num_workers=1,
    pin_memory=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_sampler=test_batch_sampler,
    num_workers=1,
    pin_memory=False,
)

## Generate erb band data and index files

In [ ]:
# input_dir = Path("/content/AEC-Challenge/datasets/synthetic/nearend_mic_signal")

# target_dir = Path("/content/AEC-Challenge/datasets/synthetic/nearend_speech")


# build_erb_store(input_dir=input_dir, target_dir=target_dir, output_dir="/content/erb_store", 
# sample_rate=sample_rate, n_fft=n_fft, hop_length=hop_length, win_length=win_length, erb_bins=erb_bins, device=device)

## Validation Function

In [24]:
@torch.no_grad()
def evaluate_model(model, loader, synthesis_matrix, device):
    model.eval()
    total_loss = 0.0
    total_batches = 0

    for batch in loader:
        input_erb = batch["input_erb"].to(device, non_blocking=True)
        input_spec = batch["input_spec"].to(device, non_blocking=True)
        target_spec = batch["target_spec"].to(device, non_blocking=True)
        predicted_gains = model(input_erb)
        predicted_spec = apply_erb_gains(input_spec, predicted_gains, synthesis_matrix)
        loss = compressed_spectral_loss(predicted_spec, target_spec)
        total_loss += loss.item()
        total_batches += 1
    return total_loss / max(total_batches, 1)


## Training Step

In [22]:
!ls -la /kaggle/working

total 12
drwxr-xr-x 3 root root 4096 Aug 28 09:55 .
drwxr-xr-x 5 root root 4096 Aug 28 07:53 ..
drwxr-xr-x 2 root root 4096 Aug 28 07:53 .virtual_documents


In [21]:
!rm -rf /kaggle/working/erb_checkpoints

In [ ]:
def train_model(model, train_loader, val_loader, train_batch_sampler, synthesis_matrix, optimizer, epochs, checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_val_loss = float("inf")
    history = []

    for epoch in range(epochs):
        train_batch_sampler.set_epoch(epoch)
        model.train()

        total_train_loss = 0.0
        total_batches = 0

        for batch in train_loader:
            input_erb = batch["input_erb"].to(device, non_blocking=True)
            input_spec = batch["input_spec"].to(device, non_blocking=True)
            target_spec = batch["target_spec"].to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            predicted_gains = model(input_erb)
            predicted_spec = apply_erb_gains(input_spec, predicted_gains, synthesis_matrix)
            loss = compressed_spectral_loss(predicted_spec, target_spec)
            loss.backward()
            optimizer.step()
            total_loss += loss.detach().item()
            total_batches += 1
            
        train_loss = (total_train_loss / max(total_batches, 1))
        val_loss = evaluate_model(
            model=model,
            loader=val_loader,
            synthesis_matrix=synthesis_matrix,
            device=device,
        )

        history.append(
            {
                "epoch": epoch+1,
                "train_loss": train_loss,
                "val_loss"; val_loss,
            }
        )
        epoch_checkpoint = checkpoint_dir / (f"epoch_{epoch + 1:03d}.ckpt")
        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_loss": train_loss,
                "val_loss": val_loss,
            },
            epoch_checkpoint,
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_checkpoint = checkpoint_dir / (f"best_model.ckpt")
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                },
                best_checkpoint,
            )
            print(
                f"Epoch {epoch + 1:02d}/{epochs:02d} "
                f"train={train_loss:.6f} "
                f"val={val_loss:.6f} "
                f"best checkpoint saved",
                flush=True,
            )
        else:
            print(
                f"Epoch {epoch + 1:02d}/{epochs:02d} "
                f"train={train_loss:.6f} "
                f"val={val_loss:.6f}",
                flush=True,
            )

    return history
            

In [13]:
model = DeepFilterNetDNN(erb_bins=erb_bins).to(device)
synthesis_matrix = erb_synthesis_matrix(filterbank).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

checkpoint_dir = "/kaggle/working/erb_checkpoints"

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    train_batch_sampler=train_batch_sampler,
    synthesis_matrix=synthesis_matrix,
    optimizer=optimizer,
    epochs=epochs,
    checkpoint_dir=checkpoint_dir,
)

Epoch: 001/010Loss: 0.188644
Epoch: 002/010Loss: 0.181524
Epoch: 003/010Loss: 0.178174
Epoch: 004/010Loss: 0.176877
Epoch: 005/010Loss: 0.175165
Epoch: 006/010Loss: 0.175038
Epoch: 007/010Loss: 0.172225
Epoch: 008/010Loss: 0.171124
Epoch: 009/010Loss: 0.171957
Epoch: 010/010Loss: 0.171721
Training completed
Checkpoints saved in: /kaggle/working/erb_checkpoints


## Testing Step

In [ ]:
checkpoint_path = checkpoint_dir / (f"checkpoint_epoch_{epochs:03d}.cpkt")
checkpoint = torch.load(checkpoint_path, map_location=device)

test_model = DeepFilterNetDNN(
    erb_bins=checkpoint["erb_bins"],
    channels=checkpoint["channels"],
    hidden_size=checkpoint["hidden_size"],
    groups=checkpoint["groups"],
).to(device)

test_model.load_state_dict(checkpoint["model_state_dict"])
test_synthesis_matrix = erb_synthesis_matrix(filterbank).to(device)



## Check whether GPU is available or not

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Test DNN

In [ ]:
sample_rate = 16000
n_fft = 512
hop_length = 128
win_length = 512
erb_bins = 32

filterbank = make_erb_filterbank(
    sample_rate=sample_rate,
    n_fft=n_fft,
    erb_bins=erb_bins,
)

input_erb = wav_to_erb(
    "/content/drive/MyDrive/nearend_mic_fileid_0.wav",
    sample_rate,
    filterbank,
    n_fft,
    hop_length,
    win_length,
)

target_erb = wav_to_erb(
    "/content/drive/MyDrive/nearend_speech_fileid_0.wav",
    sample_rate,
    filterbank,
    n_fft,
    hop_length,
    win_length,
)

num_frames = min(input_erb.shape[0], target_erb.shape[0])

input_erb = input_erb[:num_frames]
target_erb = input_erb[:num_frames]

# Add batch and channel dimensions:
# [T, E] -> [B, 1, T, E]
input_features = input_erb.unsqueeze(0).unsqueeze(0)
target_features = target_erb.unsqueeze(0).unsqueeze(0)


target_mask = target_features / (input_features + 1e-8)
target_mask = target_mask.clamp(0.0, 1.0)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepFilterNetDNN(erb_bins=erb_bins, channels=64, hidden_size=512, groups=8).to(device)

model.eval()

input_features = input_features.to(device)
target_mask = target_mask.to(device)

with torch.no_grad():
    predicted_gains = model(input_features)


print("Input:", input_features.shape)
print("Predicted gains:", predicted_gains.shape)
print("Target mask:", target_mask.shape)
print(input_features)
print(predicted_gains)

In [ ]:
wav, sr = load_mono("/content/drive/MyDrive/nearend_speech_fileid_0.wav")

print(wav)
print("Sample rate:", sr)
print("Number of samples:", wav.shape)
print("Duration:", wav.shape[0] / sr, "seconds")

magnitude = stft_magnitude(
    wav,
    n_fft=512,
    hop_length=128,
    win_length=512,
)

print("STFT shape:", magnitude.shape)